In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# Config flags
auto_verbose = False 
show_plots = False   

train_path = "./data/split/train/train.csv"
test_path  = "./data/split/test/test.csv"

# Load
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

# ==== 1. Tính price_per_m2 ====
df_train["price_per_m2"] = df_train["price"] / df_train["area"]
df_test["price_per_m2"]  = df_test["price"]  / df_test["area"]

# ==== 2. Cluster train theo price_per_m2 ====
kmeans = KMeans(n_clusters=3, random_state=42)
df_train["cluster"] = kmeans.fit_predict(df_train[["price_per_m2"]])

# ==== 3. Gán cluster cho test theo nearest centroid ====
df_test["cluster"] = kmeans.predict(df_test[["price_per_m2"]])

# ==== 4. Xuất ra 3 nhóm mô hình ====
cluster_0_train = df_train[df_train["cluster"] == 0]
cluster_1_train = df_train[df_train["cluster"] == 1]
cluster_2_train = df_train[df_train["cluster"] == 2]

cluster_0_test = df_test[df_test["cluster"] == 0]
cluster_1_test = df_test[df_test["cluster"] == 1]
cluster_2_test = df_test[df_test["cluster"] == 2]

if auto_verbose:
    print("Số lượng mỗi cluster (train):")
    print(df_train["cluster"].value_counts())
    print("\nSố lượng mỗi cluster (test):")
    print(df_test["cluster"].value_counts())

In [2]:
# Mapping địa chỉ -> cluster (mode) (ẩn log)
address_cluster_map = (
    df_train.groupby('address')['cluster']
      .agg(lambda s: s.mode().iat[0] if not s.mode().empty else -1)
      .reset_index()
      .rename(columns={'cluster':'cluster_mode'})
)
if auto_verbose:
    print('\n=== Địa chỉ thuộc cluster nào (mode) ===')
    print(address_cluster_map)

In [3]:
# Drop cột 'address' và (nếu có) 'region'; sau đó scale 'area'
from sklearn.preprocessing import StandardScaler
for df in (df_train, df_test):
    drop_cols = [c for c in ['address','region'] if c in df.columns]
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

scaler_area = StandardScaler()
df_train['area'] = scaler_area.fit_transform(df_train[['area']])
df_test['area']  = scaler_area.transform(df_test[['area']])

In [4]:
import torch

def build_dataset(df):
    feature_cols = ["area", "bedrooms", "bathrooms"]
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
    y = torch.tensor(df[["price"]].values, dtype=torch.float32)
    return X, y

X0_train, y0_train = build_dataset(cluster_0_train)
X1_train, y1_train = build_dataset(cluster_1_train)
X2_train, y2_train = build_dataset(cluster_2_train)

X0_test, y0_test = build_dataset(cluster_0_test)
X1_test, y1_test = build_dataset(cluster_1_test)
X2_test, y2_test = build_dataset(cluster_2_test)

In [5]:
import torch.nn as nn
from sklearn.metrics import r2_score, mean_squared_error

def train_cluster_model(X_train, y_train, epochs=3000, lr=1e-3):
    model = nn.Linear(X_train.shape[1], 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for ep in range(epochs):
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model


def evaluate_cluster_model(model, X_test, y_test):
    with torch.no_grad():
        y_pred = model(X_test).cpu().numpy()
    y_true = y_test.cpu().numpy()
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return r2, rmse

In [6]:
models = {}

for idx, (Xt_tr, Yt_tr, Xt_te, Yt_te) in enumerate([
    (X0_train, y0_train, X0_test, y0_test),
    (X1_train, y1_train, X1_test, y1_test),
    (X2_train, y2_train, X2_test, y2_test)
]):
    model = train_cluster_model(Xt_tr, Yt_tr)

    r2, rmse = evaluate_cluster_model(model, Xt_te, Yt_te)

    models[idx] = model
    if auto_verbose:
        print(f"Cluster {idx}: R² = {r2:.4f} | RMSE = {rmse:.4f}")

In [7]:
# Notebook-based progress display
from IPython.display import display, clear_output
import time

def train_cluster_model_verbose(X_train, y_train, epochs=1000, lr=1e-3, log_every=100, title=""):
    model = nn.Linear(X_train.shape[1], 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for ep in range(1, epochs+1):
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if ep % log_every == 0 or ep in (1, epochs):
            clear_output(wait=True)
            display(f"{title} Epoch {ep}/{epochs} - Loss: {loss.item():.6f}")
            time.sleep(0.01)
    return model


In [ ]:
from ipywidgets import VBox, HBox, FloatText, IntText, Dropdown, Button, Output, Layout
from IPython.display import display, clear_output
import numpy as np
import torch

# Danh sách địa chỉ dựa trên bảng mapping (tạo ở cell trước)
if 'address_cluster_map' in globals():
    address_list = sorted(address_cluster_map['address'].dropna().unique().tolist())
else:
    # Fallback nếu chưa có mapping (ít gặp): lấy từ df_train nếu còn cột address
    address_list = sorted(df_train['address'].dropna().unique().tolist()) if 'address' in df_train.columns else []

# Tạo dict map địa chỉ -> cluster_mode
addr_to_cluster = {}
if 'address_cluster_map' in globals():
    addr_to_cluster = dict(zip(address_cluster_map['address'], address_cluster_map['cluster_mode']))

# Widgets (xếp dọc thay vì ngang)
address_w = Dropdown(options=address_list, description='Address:', layout=Layout(width='400px'))
area_w    = FloatText(value=float(df_train['area'].median()) if 'area' in df_train.columns else 50.0, description='Area:', layout=Layout(width='300px'))
bed_w     = IntText(value=int(df_train['bedrooms'].median()) if 'bedrooms' in df_train.columns else 2, description='Bedrooms:', layout=Layout(width='300px'))
bath_w    = IntText(value=int(df_train['bathrooms'].median()) if 'bathrooms' in df_train.columns else 2, description='Bathrooms:', layout=Layout(width='300px'))
run_btn   = Button(description='Predict', button_style='primary', layout=Layout(width='150px'))
out       = Output()

# Handler
@run_btn.on_click
def _on_run(_):
    with out:
        clear_output(wait=True)
        if 'models' not in globals() or not models:
            print('Models chưa sẵn sàng. Hãy chạy toàn bộ notebook trước.')
            return
        # Xác định cluster từ địa chỉ; nếu không có -> gán 1
        sel_addr = address_w.value
        cluster_id = addr_to_cluster.get(sel_addr, 1)
        # Nếu model cụ thể không tồn tại, fallback về 1 nếu có
        if cluster_id not in models:
            if 1 in models:
                cluster_id = 1
            else:
                # nếu vẫn không có, lấy cluster đầu tiên có model
                cluster_id = sorted(models.keys())[0]
        # scale area (nếu scaler sẵn có)
        try:
            scaled_area = scaler_area.transform(np.array([[area_w.value]])).ravel()[0]
        except Exception:
            scaled_area = area_w.value
        x_np = np.array([[scaled_area, float(bed_w.value), float(bath_w.value)]], dtype=np.float32)
        x_t = torch.tensor(x_np, dtype=torch.float32)
        with torch.no_grad():
            yhat = models[cluster_id](x_t).cpu().numpy().ravel()[0]
        print(f"Address: {sel_addr} -> Cluster {cluster_id}")
        print(f"Predicted price = {yhat:,.2f}")

# UI dọc
ui = VBox([
    address_w,
    area_w,
    bed_w,
    bath_w,
    run_btn,
    out
])
display(ui)